# Template Based Diagram Layouts
This notebook is intended to explore the use of template-based diagram layouts 
for visualizing relationships between structures in a dataset. 
The goal is to compare this approach with the current optimization-based layout 
and also to evaluate the use to Networkx for layouts.

## Setup

### Imports

In [1]:
from typing import Dict, List, Union

from pathlib import Path
import re

from pprint import pprint

In [2]:
import xml.etree.ElementTree as ET
from itertools import chain


In [3]:
import pandas as pd
import xlwings as xw
import networkx as nx
from matplotlib import pyplot as plt

In [4]:
from structure_set import StructureSet
from dicom import DicomStructureFile
from structure_id_parser import parse_structure_metadata, merge_priority_columns

INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Paths

In [5]:
base_path = Path.cwd()
dicom_path = base_path / "tests"
structure_filter_def = base_path / "src" / "webapp" / "config" / "structure_filter_rules.json"

## Utility Functions

### Load the DICOM file

In [6]:
def load_structures(dicom_file: DicomStructureFile,
                    apply_filter=False)->pd.DataFrame:
    '''
    Load structure names from a DICOM file and optionally apply a filter.

    Args:
        dicom_file (DicomStructureFile): The DICOM structure file.
        apply_filter (bool): Whether to apply a filter to the structures.
            Default is False.

    Returns:
        pd.DataFrame: A DataFrame containing the structure IDs and their metadata.
    '''
    # get structure IDs
    meta_data = dicom_file.get_structure_filter_metadata()
    if apply_filter:
        filter_report = dicom_file.evaluate_structure_filters(structure_filter_def)
        selection = filter_report.SelectedByDefault & filter_report.DisplayedByDefault
        meta_data = meta_data[selection]
    return meta_data

## Load an example CNS DICOM structure set

### Path to the example DICOM file

In [7]:
dcm_file_name = 'RS.CNS_FSRT_2_GTV.BRAI.dcm'
dcm_file_path = dicom_path / dcm_file_name
dicom_file = DicomStructureFile(top_dir=dicom_path, file_path=dcm_file_path)
pprint(dicom_file.structure_set_info)


INFO:dicom:Successfully loaded DICOM dataset from RS.CNS_FSRT_2_GTV.BRAI.dcm
INFO:dicom:Extracted 2911 contours from 35 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.CNS_FSRT_2_GTV.BRAI.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.040 cm/pixel


{'File': WindowsPath("d:/OneDrive - Queen's University/Python/Projects/StructureRelations/tests/RS.CNS_FSRT_2_GTV.BRAI.dcm"),
 'PatientID': 'CNS_FSRT_2_GTV',
 'PatientLastName': 'CNS_FSRT_2_GTV_Partition_Error',
 'PatientName': 'CNS_FSRT_2_GTV_Partition_Error',
 'SeriesDescription': '',
 'SeriesNumber': '0',
 'StructureSet': 'BRAI',
 'StudyID': ''}


In [8]:
structure_data = load_structures(dicom_file)
structures_df = parse_structure_metadata(structure_data)

### Create a grouping column from the target number, dose, classifier and organ columns

In [ ]:
horizontal_group_columns = ['TargetNumber', 'TargetDose', 'Classifier',
                            'Combined', 'TargetOrgan']
horizontal_columns_to_merge = [col for col in horizontal_group_columns
                               if col in structures_df.columns]

vertical_group_columns = ['Mod', 'DICOM Type', 'TargetType',
                          'ExpansionSize', 'Structure Code']
vertical_columns_to_merge = [col for col in vertical_group_columns
                             if col in structures_df.columns]

structures_df['h_grouping'] = merge_priority_columns(structures_df,
                                                     horizontal_columns_to_merge)
structures_df['v_grouping'] = merge_priority_columns(structures_df,
                                                     vertical_columns_to_merge)

structures_df.sort_values(by=['h_grouping', 'v_grouping'], inplace=True)

include_structures = list(structures_df.index)

In [10]:
structures_df[['h_grouping', 'v_grouping']]

,h_grouping,v_grouping
Structure,,
GTV 1 xxGy,1,GTV
PTV 1_24Gyin3,1,PTV
shell_PTV 1_24Gyin3,1,shell
GTV 2 xxGy,2,GTV
PTV 2_24Gyin3,2,PTV
shell_PTV 2_24Gyin3,2,shell
GTV Total,Total,GTV
PTV Total,Total,PTV
CTV,NaN,CTV


### Determine the relationships between the structures

In [11]:
structure_set = StructureSet(dicom_structure_file=dicom_file,
                             include_structures=include_structures)


INFO:structure_set:Building StructureSet from 2911 contour points (unit: cm)
INFO:structure_set:Skipping structure BODY (1) due to filters
INFO:structure_set:Adding structure GTV 1 xxGy (4)
INFO:structure_set:Adding structure GTV 2 xxGy (5)
INFO:structure_set:Adding structure GTV Total (9)
INFO:structure_set:Adding structure PTV 1_24Gyin3 (10)
INFO:structure_set:Adding structure PTV 2_24Gyin3 (11)
INFO:structure_set:Adding structure PTV Total (15)
INFO:structure_set:Skipping structure Avoid INNER (17) due to filters
INFO:structure_set:Skipping structure Avoid MID (18) due to filters
INFO:structure_set:Skipping structure Avoid OUTER (19) due to filters
INFO:structure_set:Skipping structure Brain (20) due to filters
INFO:structure_set:Skipping structure BrainStem (21) due to filters
INFO:structure_set:Skipping structure Cochlea L (22) due to filters
INFO:structure_set:Skipping structure Cochlea R (23) due to filters
INFO:structure_set:Skipping structure Globe L (24) due to filters
INFO:s

In [12]:
relations = structure_set.relationship_summary()
relations
relation_dict = {
    (col_idx, row_idx): relations.at[row_idx, col_idx]
    for row_idx in relations.index
    for col_idx in relations.columns
}


In [13]:
structure_set.summary().set_index('ROI')['Name'].to_dict()

{4: 'GTV 1 xxGy',
 5: 'GTV 2 xxGy',
 9: 'GTV Total',
 10: 'PTV 1_24Gyin3',
 11: 'PTV 2_24Gyin3',
 15: 'PTV Total',
 34: 'PTV+15',
 59: 'shell_PTV 2_24Gyin3',
 62: 'shell_PTV 1_24Gyin3'}

In [14]:
structure_set.get_relationship(34, 62).relationship_type.label

'Overlaps with'